``SequenceFeature.kmer_composition`` builds a **k-mer-composition baseline**: for each sequence it concatenates the selected Parts into one span and returns the fraction of every ordered overlapping k-mer of adjacent canonical residues, a ``(n_seq, 20 ** k)`` matrix ``X``. It carries no positional information (unlike :meth:`CPP.run` / :meth:`feature_matrix`), so it is the natural **baseline to compare CPP against**.

``k`` selects the composition: ``k=1`` is amino-acid composition (**AAC**, 20 columns, identical to :meth:`aa_composition`), ``k=2`` dipeptide composition (**DPC**, 400 columns, identical to :meth:`dipeptide_composition`), and higher ``k`` captures longer local sequential order.

In [1]:
import numpy as np
import pandas as pd
import aaanalysis as aa
aa.options["verbose"] = False

# Six short sequences (whole sequence used as the span; no flanks needed)
df_seq = pd.DataFrame({
    "entry": [f"P{i}" for i in range(6)],
    "sequence": ["ACDEFGHIKLMNPQRSTVWY", "MKKLLAACDE", "ACACACAC",
                 "WYWYWYWY", "GGGGSSSS", "LLKKLLKK"],
})
sf = aa.SequenceFeature()
aa.display_df(df_seq, n_rows=10, show_shape=True)

DataFrame shape: (6, 2)


,entry,sequence
1,P0,ACDEFGHIKLMNPQRSTVWY
2,P1,MKKLLAACDE
3,P2,ACACACAC
4,P3,WYWYWYWY
5,P4,GGGGSSSS
6,P5,LLKKLLKK


**`k` — the composition order.** `k=1` gives AAC (20 columns): the fraction of each amino acid.

In [2]:
X_aac = sf.kmer_composition(df_seq=df_seq, k=1, return_df=True)
print(f"Shape: {X_aac.shape}")
aa.display_df(X_aac.round(3), n_rows=10, show_shape=True)

Shape: (6, 20)
DataFrame shape: (6, 20)


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
entry,,,,,,,,,,,,,,,,,,,,
P0,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000
P1,0.200000,0.100000,0.100000,0.100000,0.000000,0.000000,0.000000,0.000000,0.200000,0.200000,0.100000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
P2,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
P3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.500000
P4,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000
P5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


`k=2` gives DPC (400 columns): the fraction of each ordered adjacent residue pair `AA, AC, ..., YY`.

In [3]:
X_dpc = sf.kmer_composition(df_seq=df_seq, k=2, return_df=True)
print(f"Shape: {X_dpc.shape}")
# Show only the non-zero dipeptide columns for readability
nz = X_dpc.loc[:, (X_dpc.fillna(0) != 0).any(axis=0)]
aa.display_df(nz.round(3), n_rows=10, show_shape=True)

Shape: (6, 400)
DataFrame shape: (6, 30)


,AA,AC,CA,CD,DE,EF,FG,GG,GH,GS,HI,IK,KK,KL,LA,LK,LL,LM,MK,MN,NP,PQ,QR,RS,SS,ST,TV,VW,WY,YW
entry,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
P0,0.000000,0.053000,0.000000,0.053000,0.053000,0.053000,0.053000,0.000000,0.053000,0.000000,0.053000,0.053000,0.000000,0.053000,0.000000,0.000000,0.000000,0.053000,0.000000,0.053000,0.053000,0.053000,0.053000,0.053000,0.000000,0.053000,0.053000,0.053000,0.053000,0.000000
P1,0.111000,0.111000,0.000000,0.111000,0.111000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.111000,0.111000,0.111000,0.000000,0.111000,0.000000,0.111000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
P2,0.000000,0.571000,0.429000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
P3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.571000,0.429000
P4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.429000,0.000000,0.143000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.429000,0.000000,0.000000,0.000000,0.000000,0.000000
P5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.286000,0.143000,0.000000,0.286000,0.286000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


`k=3` gives tripeptide composition (8000 columns) — longer local order at the cost of a much wider, sparser matrix. `k` is capped at 4 (`20 ** k` grows exponentially).

In [4]:
X_k3 = sf.kmer_composition(df_seq=df_seq, k=3)
print(f"k=3 shape: {X_k3.shape}  (20**3 = {20**3} columns)")
print(f"Each row sums to 1 (spans with >= 3 residues): "
      f"{np.allclose(np.nansum(X_k3, axis=1), 1.0)}")

k=3 shape: (6, 8000)  (20**3 = 8000 columns)
Each row sums to 1 (spans with >= 3 residues): True


**`k=1` and `k=2` are exactly the named baselines.** `kmer_composition(k=1)` equals :meth:`aa_composition` and `kmer_composition(k=2)` equals :meth:`dipeptide_composition` — the named methods are convenience aliases for those two orders.

In [5]:
same_aac = np.array_equal(sf.kmer_composition(df_seq=df_seq, k=1),
                          sf.aa_composition(df_seq=df_seq), equal_nan=True)
same_dpc = np.array_equal(sf.kmer_composition(df_seq=df_seq, k=2),
                          sf.dipeptide_composition(df_seq=df_seq), equal_nan=True)
print(f"k=1 == aa_composition: {same_aac}")
print(f"k=2 == dipeptide_composition: {same_dpc}")

k=1 == aa_composition: True
k=2 == dipeptide_composition: True


**`list_parts` — which span to count over.** By default the whole `tmd_jmd` span is used; pass specific Parts (needs TMD/JMD columns) to restrict the composition to, e.g., the TMD only.

In [6]:
df_seq_tmd = df_seq.assign(tmd_start=2, tmd_stop=7)  # TMD within the shortest sequence (len 8)
X_tmd = sf.kmer_composition(df_seq=df_seq_tmd, k=1, list_parts="tmd", return_df=True)
aa.display_df(X_tmd.round(3), n_rows=10, show_shape=True)

DataFrame shape: (6, 20)


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
entry,,,,,,,,,,,,,,,,,,,,
P0,0.000000,0.167000,0.167000,0.167000,0.167000,0.167000,0.167000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
P1,0.333000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.333000,0.333000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
P2,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
P3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.500000
P4,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000
P5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


**`return_df`.** With `return_df=False` (default) a plain `np.ndarray` is returned for direct use as a model matrix `X`; with `return_df=True` a labeled `DataFrame` (one column per k-mer) is returned. A span with fewer than `k` canonical residues has an all-`NaN` row.

In [7]:
X_arr = sf.kmer_composition(df_seq=df_seq, k=2)
print(type(X_arr).__name__, X_arr.shape, X_arr.dtype)

ndarray (6, 400) float64


**Getting CPP-ready scales / categories (`return_scales`).** For the feature map you need a scale set + a category table, not the composition matrix `X`. `return_scales=True` makes the return the 3-tuple `(X, df_scales, df_cat)`. For **k=1** `df_scales` is the 20×20 one-hot identity — feed it with `df_cat` to `aa.CPP(df_parts, df_scales=df_scales, df_cat=df_cat, split_kws=<Segment(1,1)>).run(...)` to obtain amino-acid composition as a real `df_feat` / feature map. For **k>=2** `df_scales` is `None` (a k-mer is not a per-residue scale) and `df_cat` labels each k-mer by its residue class.

In [8]:
X, df_scales, df_cat = sf.kmer_composition(df_seq=df_seq, k=1, return_scales=True)
print(f"k=1 df_scales (one-hot identity): {df_scales.shape}")
aa.display_df(df_cat, n_rows=10, show_shape=True)

k=1 df_scales (one-hot identity): (20, 20)
DataFrame shape: (20, 5)


,scale_id,category,subcategory,scale_name,scale_description
1,A,Nonpolar,Nonpolar,A,Amino acid A indicator
2,C,Polar,Polar,C,Amino acid C indicator
3,D,Negative,Negative,D,Amino acid D indicator
4,E,Negative,Negative,E,Amino acid E indicator
5,F,Aromatic,Aromatic,F,Amino acid F indicator
6,G,Nonpolar,Nonpolar,G,Amino acid G indicator
7,H,Positive,Positive,H,Amino acid H indicator
8,I,Nonpolar,Nonpolar,I,Amino acid I indicator
9,K,Positive,Positive,K,Amino acid K indicator
10,L,Nonpolar,Nonpolar,L,Amino acid L indicator
